# Getting started

In this notebook you will learn how to

- describe your data with two metadata files
- load a collection of supply and use tables (SUTs)
- find codes and select rows
- make your first inspection
- write the tables to files

The data files are described in the [README](README.md).

In [ ]:
from pathlib import Path

import pandas as pd
import sutlab as sl

data_folder = Path("data")

## Metadata

`sutlab` does not assume any particular column names. Two Excel files tell it how your data is organised:

1. **The columns file** says which column plays which role.
2. **The classifications file** holds labels and other information about the codes, with one sheet per topic.

You can call the files what you like. Errors in them give a message that says what is wrong.

### The columns file

This is how the file looks in Excel. `column` is the name in your data, and `role` is what the column means. Each role and each column may only appear once.

In [ ]:
pd.read_excel(data_folder / "ta_columns.xlsx")

These roles are **required**:

| Role | Meaning |
|---|---|
| `id` | The year, or whatever else separates the tables in a collection. Files with one year each do not contain this column, but it needs a name |
| `product` | Product |
| `transaction` | Transaction code |
| `category` | Industry or consumption group |
| `price_basic` | Value in basic prices |
| `price_purchasers` | Value in purchasers' prices |

These are **optional**. Include the price layers that you have in your data:

| Role | Meaning |
|---|---|
| `wholesale_margins` | Wholesale margins |
| `retail_margins` | Retail margins |
| `product_taxes_less_subsidies` | Taxes less subsidies on products, excluding VAT |
| `vat` | VAT |
| `trade_margins` | Wholesale and retail margins together, if they are not split |
| `transport_margins` | Transport margins |
| `product_taxes` | Taxes on products, if subsidies are recorded separately |
| `product_subsidies` | Subsidies on products, if recorded separately |

Price layer rates can only be computed for the first four. Inspections that include such rates give an error if the data has any of the others.

### The classifications file

Only the sheet `transactions` is required. The others can be left out, but then you get no labels. The codes are in columns named as in your data, and the labels in the same name with `_txt` added. With our names:

| Sheet | Columns | Content |
|---|---|---|
| `transactions` | `trans`, `trans_txt`, `table`, `esa_code`, optionally `gdp_decomp` | Transactions, see below |
| `products` | `nrnr`, `nrnr_txt` | Products |
| `industries` | `brch`, `brch_txt` | Industries |
| `individual_consumption` | `brch`, `brch_txt` | Groups of individual consumption |
| `collective_consumption` | `brch`, `brch_txt` | Groups of collective consumption |
| `margin_products` | `nrnr`, `price_layer`, optionally `nrnr_txt` | Products whose supply is a trade margin, and the price layer column they belong to (for example `eng`). They are left out when looking for unbalanced products |
| `unspec_industries` | `brch`, optionally `brch_txt` | Unspecified industries. They are left out of output, intermediate consumption, value added and the industry tables when computing GDP |
| `classifications` | `dimension`, `classification` | The name of the classification used for each of the dimensions |

These are the sheets in our file:

In [ ]:
pd.ExcelFile(data_folder / "ta_classifications.xlsx").sheet_names

The sheet `transactions` has these columns besides the code and the label:

- `table`: whether the transaction belongs to `supply` or `use`
- `esa_code`: what kind of transaction it is, in a standard terminology
- `gdp_decomp`: the row of the GDP table it belongs to. Only needed to compute GDP

The possible values of `esa_code` are:

| Code | Meaning |
|---|---|
| `P1` | Output |
| `P2` | Intermediate consumption |
| `P3`, `P31`, `P32` | Final, individual and collective consumption |
| `P51g` | Gross fixed capital formation |
| `P52`, `P53` | Changes in inventories, acquisitions less disposals of valuables |
| `P6`, `P7` | Exports, imports |
| `D2121` | Import duties |

The category of a row is looked up among the industries for `P1` and `P2`, individual consumption for `P31` and collective consumption for `P32`.

### Load the metadata

Now we load both files.

In [ ]:
metadata = sl.load_metadata_from_excel(
    data_folder / "ta_columns.xlsx",
    data_folder / "ta_classifications.xlsx",
)

The result holds the roles and the classifications. For example, this is the name of the product column:

In [ ]:
metadata.columns.product

And this is the sheet with the transactions:

In [ ]:
metadata.classifications.transactions

## Load a collection of SUTs

Each parquet file holds one year, and contains both supply and use. We give the years, the paths, the metadata and the price basis.

There are similar functions for CSV and Excel files, and for files where all years are stored together. Type `sl.load_sut_` and press Tab to see them.

In [ ]:
years = [2016, 2017, 2018, 2019]
current_paths = [data_folder / f"ta_l_{year}.parquet" for year in years]

sut = sl.load_sut_from_separated_parquet(
    years,
    current_paths,
    metadata,
    "current_year",
    print_paths=True,
)

Statistics Denmark's own fixed-width files can be loaded directly from the network drives. This is not run here. Replace `"x"` with the letter of the drive you use:

```python
paths = sl.make_final_annual_sut_paths(years, "x", "current_year")
sut = sl.load_sut_from_separated_nr_legacy_fwf(years, paths, metadata, "current_year")
```

## What is in a SUT?

A `SUT` holds two ordinary pandas DataFrames, `supply` and `use`, with all years stacked on top of each other. Each row is one cell of the matrix: a product, a transaction and a category (an industry or a consumption group). Imports, exports and investment have an empty category.

Supply is in basic prices only.

In [ ]:
sut.supply.head()

Use has all the price columns, from basic prices over the price layers (margins, taxes and VAT) to purchasers' prices.

In [ ]:
sut.use.head()

## Find codes

The `get_` functions list the codes in the data together with their labels.

In [ ]:
sut.get_ids()

In [ ]:
sut.get_codes_transactions()

You can filter with a pattern. `*` matches anything, so this gives all products that start with `V22`.

In [ ]:
sut.get_codes_products("V22*")

## Select rows

`filter_rows` returns a new, smaller SUT. The original is not changed. This is true for all functions in `sutlab`: they return something new, so remember to save the result in a variable.

In [ ]:
wine = sut.filter_rows(products="V220400", ids=2019)
wine.use

The same patterns work everywhere a function filters on codes:

| Pattern | Meaning |
|---|---|
| `"V220400"` | exactly this code |
| `["V220400", "V220800"]` | any of these codes |
| `"V22*"` | codes starting with V22 |
| `"V220400:V220800"` | codes from V220400 to V220800, both included |
| `"~V22*"` | all codes except those starting with V22 |

`V22` is drinks, apart from `V220900` which is vinegar. Here we select household consumption of drinks without the vinegar.

In [ ]:
drinks = sut.filter_rows(
    products=["V22*", "~V220900"],
    transactions="3110",
)
drinks.get_codes_products()

## A first inspection

Functions that start with `inspect_` give an overview of the data as a set of tables. Here is the balance between supply and use for one product.

In [ ]:
inspection = sut.inspect_products("V220400")
inspection.balance

## Write to files

The tables can be written to parquet, CSV or Excel. Here we save the small SUT from before as a single Excel file.

In [ ]:
output_folder = Path("output")
output_folder.mkdir(exist_ok=True)

wine.write_to_combined_excel(output_folder / "wine_2019.xlsx")

## Getting help

Put a question mark in front of a function to read its documentation, for example `?sut.filter_rows`.

---
Next: [02_inspecting_a_sut](02_inspecting_a_sut.ipynb)